# 🔥 Forest Fire EDA — Beginner Level
**Dataset:** Brazilian Amazon Forest Fires (1998–2017)  
**Columns:** `year`, `state`, `month`, `number` (fires reported), `date`

---

## Code 1 — Load the Data & First Look

In [ ]:
import pandas as pd
import numpy as np

# Load both files with latin-1 encoding (fixes UnicodeDecodeError)
df = pd.read_csv('forest_fire.csv', encoding='latin-1')
df2 = pd.read_csv('forest_fire_2.csv', encoding='latin-1')

print('Shape of dataset:', df.shape)         # (rows, columns)
print('\nColumn names:', df.columns.tolist())
print('\nData Types:\n', df.dtypes)
print('\nFirst 5 rows:')
df.head()

**What this does:**  
- Imports the two main libraries — `pandas` for data handling, `numpy` for math  
- Loads the CSV with `encoding='latin-1'` to avoid the `UnicodeDecodeError`  
- Checks the shape, column names, and data types before doing anything else  

---

## Code 2 — Summary Statistics with NumPy & Pandas

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

fires = df['number'].values   # convert column to numpy array

print('=== Pandas describe() ===')
print(df['number'].describe())

print('\n=== NumPy calculations ===')
print('Mean   :', np.mean(fires).round(2))
print('Median :', np.median(fires))
print('Std Dev:', np.std(fires).round(2))
print('Min    :', np.min(fires))
print('Max    :', np.max(fires))
print('Range  :', np.ptp(fires))   # peak-to-peak = max - min

**What this does:**  
- `describe()` gives a quick pandas summary in one line  
- `np.mean`, `np.median`, `np.std` compute key statistics manually using NumPy  
- `np.ptp()` (peak-to-peak) returns the range — useful to spot spread in data  

---

## Code 3 — Check for Missing Values & Duplicates

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

print('=== Missing Values per Column ===')
print(df.isnull().sum())

print('\nTotal missing values:', df.isnull().sum().sum())
print('Missing % per column:')
print((df.isnull().sum() / len(df) * 100).round(2))

print('\n=== Duplicate Rows ===')
print('Total duplicates:', df.duplicated().sum())

# Drop duplicates if any exist
df_clean = df.drop_duplicates()
print('Shape after removing duplicates:', df_clean.shape)

**What this does:**  
- `isnull().sum()` counts missing values — always check this before analysis  
- Calculates the **percentage** of missing data, more informative than raw counts  
- `duplicated().sum()` finds exact duplicate rows that can skew results  

---

## Code 4 — Fires by Year (Trend Analysis)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

# Total fires reported per year
fires_by_year = df.groupby('year')['number'].sum().reset_index()
fires_by_year.columns = ['year', 'total_fires']

print('Total fires per year:')
print(fires_by_year.to_string(index=False))

# Year with most and least fires
max_year = fires_by_year.loc[fires_by_year['total_fires'].idxmax()]
min_year = fires_by_year.loc[fires_by_year['total_fires'].idxmin()]

print('\nYear with MOST fires :', max_year['year'], '->', int(max_year['total_fires']))
print('Year with LEAST fires:', min_year['year'], '->', int(min_year['total_fires']))

# Numpy: year-over-year change
yoy_change = np.diff(fires_by_year['total_fires'].values)
print('\nYear-over-Year Change in fires (NumPy):')
print(yoy_change)

**What this does:**  
- `groupby` + `sum()` aggregates total fires per year — fundamental EDA operation  
- `idxmax()` and `idxmin()` locate the best and worst years instantly  
- `np.diff()` calculates the year-over-year difference to spot trends/spikes  

---

## Code 5 — Fires by State (Top & Bottom States)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

fires_by_state = df.groupby('state')['number'].sum().sort_values(ascending=False)

print('=== Top 5 States with Most Fires ===')
print(fires_by_state.head())

print('\n=== Bottom 5 States with Least Fires ===')
print(fires_by_state.tail())

# Percentage contribution of each state
state_pct = (fires_by_state / fires_by_state.sum() * 100).round(2)
print('\n=== % Share of Total Fires by State ===')
print(state_pct)

# NumPy: percentile-based classification
values = fires_by_state.values
p75 = np.percentile(values, 75)
print(f'\nStates above 75th percentile (high risk): {np.sum(values > p75)} states')

**What this does:**  
- Groups and sorts by state to rank fire frequency across Brazil  
- Calculates **percentage share** so you can compare states fairly  
- `np.percentile()` classifies states as high-risk based on the 75th percentile  

---

## Code 6 — Monthly Fire Pattern (Seasonality)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

# Portuguese month names → month number for sorting
month_order = {
    'Janeiro':1, 'Fevereiro':2, 'Março':3, 'Abril':4,
    'Maio':5, 'Junho':6, 'Julho':7, 'Agosto':8,
    'Setembro':9, 'Outubro':10, 'Novembro':11, 'Dezembro':12
}

df['month_num'] = df['month'].map(month_order)
fires_by_month = df.groupby(['month_num', 'month'])['number'].sum().reset_index()
fires_by_month = fires_by_month.sort_values('month_num')

print('Average fires per month across all years:')
print(fires_by_month[['month', 'number']].to_string(index=False))

# NumPy: which months are above average?
monthly_avg = np.mean(fires_by_month['number'].values)
above_avg = fires_by_month[fires_by_month['number'] > monthly_avg]['month'].tolist()
print(f'\nMonthly average: {monthly_avg:.0f} fires')
print('Months ABOVE average (peak fire season):', above_avg)

**What this does:**  
- Maps Portuguese month names to numbers so they sort correctly  
- Reveals seasonal patterns — crucial for fire risk planning  
- `np.mean()` on grouped data identifies which months are above average  

---

## Code 7 — Filtering, Slicing & Conditional Selection

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

# Filter: only rows where fires > 500
extreme_fires = df[df['number'] > 500]
print(f'Records with fires > 500: {len(extreme_fires)}')
print(extreme_fires.sort_values('number', ascending=False).head(10))

# Filter: specific state and year range
mato_grosso = df[(df['state'] == 'Mato Grosso') & (df['year'] >= 2010)]
print(f'\nMato Grosso records from 2010 onwards: {len(mato_grosso)}')

# NumPy: boolean mask for same operation
numbers = df['number'].values
mask = numbers > np.percentile(numbers, 90)   # top 10% of fire events
print(f'\nTop 10% extreme fire events (NumPy mask): {mask.sum()} records')
print('Min fires to be in top 10%:', np.percentile(numbers, 90))

**What this does:**  
- Boolean filtering with `df[condition]` — one of the most used pandas patterns  
- Combines multiple conditions using `&` (AND) operator  
- NumPy boolean mask achieves the same but directly on arrays  

---

## Code 8 — Correlation & NumPy Linear Algebra

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

# Pivot: states as rows, years as columns, values = total fires
pivot = df.groupby(['state', 'year'])['number'].sum().unstack(fill_value=0)

# Correlation between years — do fire patterns repeat?
corr_matrix = pivot.T.corr()    # correlate year pairs
print('Correlation between years (first 5x5):')
print(corr_matrix.iloc[:5, :5].round(2))

# NumPy: correlation between two specific years
year_2015 = pivot[2015].values.astype(float)
year_2016 = pivot[2016].values.astype(float)

corr = np.corrcoef(year_2015, year_2016)[0, 1]
print(f'\nNumPy Correlation between 2015 and 2016 fire patterns: {corr:.4f}')
print('Interpretation:', 'Strong positive correlation' if corr > 0.7 else 'Weak correlation')

**What this does:**  
- `pivot` reshapes data to a state × year matrix — great for comparison  
- Pandas `.corr()` on the transpose finds which years had similar fire patterns  
- `np.corrcoef()` computes correlation between two specific years as arrays  

---

## Code 9 — Adding New Columns & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('forest_fire.csv', encoding='latin-1')

# 1. Log-transform the fire count (useful for skewed data)
df['log_fires'] = np.log1p(df['number'])   # log1p handles zeros safely

# 2. Classify fire severity
def classify_fire(n):
    if n == 0:     return 'None'
    elif n < 50:   return 'Low'
    elif n < 200:  return 'Medium'
    else:          return 'High'

df['severity'] = df['number'].apply(classify_fire)

# 3. Convert date column to datetime & extract quarter
df['date'] = pd.to_datetime(df['date'])
df['quarter'] = df['date'].dt.quarter

print('New columns added:')
print(df[['year', 'state', 'number', 'log_fires', 'severity', 'quarter']].head(10))

print('\nSeverity distribution:')
print(df['severity'].value_counts())

**What this does:**  
- `np.log1p()` applies log-transformation — reduces skewness in fire counts  
- `.apply()` with a custom function creates a categorical severity column  
- `pd.to_datetime()` + `.dt.quarter` extracts seasonal quarters from dates  

---

## Code 10 — Merging Both Files & Comparing Datasets

In [ ]:
import pandas as pd
import numpy as np

df1 = pd.read_csv('forest_fire.csv',   encoding='latin-1')
df2 = pd.read_csv('forest_fire_2.csv', encoding='latin-1')

# Tag each source
df1['source'] = 'file_1'
df2['source'] = 'file_2'

# Combine both files vertically
combined = pd.concat([df1, df2], ignore_index=True)
print('Combined shape:', combined.shape)

# Compare total fires per source
print('\nTotal fires by source:')
print(combined.groupby('source')['number'].sum())

# Compare per-year averages using NumPy
mean_f1 = np.mean(df1['number'].values)
mean_f2 = np.mean(df2['number'].values)
print(f'\nAvg fires — file_1: {mean_f1:.2f} | file_2: {mean_f2:.2f}')

# Check if both datasets cover same years
years_f1 = set(df1['year'].unique())
years_f2 = set(df2['year'].unique())
print('Years only in file_1:', years_f1 - years_f2)
print('Years only in file_2:', years_f2 - years_f1)
print('Common years:', len(years_f1 & years_f2))

**What this does:**  
- `pd.concat()` stacks both CSVs into one combined dataset  
- Source tagging lets you compare file_1 vs file_2 in a single grouped analysis  
- Set operations (`-`, `&`) on unique years quickly reveal coverage differences  

---

## Summary Table

| # | Topic | Key Functions |
|---|-------|---------------|
| 1 | Load Data | `read_csv`, `head`, `dtypes` |
| 2 | Summary Stats | `describe`, `np.mean/median/std` |
| 3 | Missing & Duplicates | `isnull`, `duplicated`, `drop_duplicates` |
| 4 | Yearly Trend | `groupby`, `sum`, `np.diff` |
| 5 | State Analysis | `sort_values`, `np.percentile` |
| 6 | Seasonality | `map`, `groupby`, `np.mean` |
| 7 | Filtering | Boolean indexing, NumPy masks |
| 8 | Correlation | `corr`, `unstack`, `np.corrcoef` |
| 9 | Feature Engineering | `apply`, `np.log1p`, `dt.quarter` |
| 10 | Merging Files | `pd.concat`, set operations |
